In [1]:
import pandas as pd
import numpy as np
from grape import Graph
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Edge type prediction with NP-KG

### Load KG

In [2]:
KG_PATH = '../../np-kg/resources/knowledge_graphs/'
EMBED_PATH = 'embeddings/'
DATA_PATH = '../data/'
DIR_PATH = 'results/'

In [3]:
npkg = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v3.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v3.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [4]:
npkg.get_number_of_connected_components()

(348, 1, 1089125)

In [5]:
npkg = npkg.remove_disconnected_nodes()

In [6]:
npkg.enable()

In [9]:
##get negative edges
n_edges = npkg.get_number_of_directed_edges()
n_neg_edges = int(n_edges*0.2)
n_neg_edges

1557384

In [10]:
negative_graph = npkg.sample_negative_graph(
    number_of_negative_samples=n_neg_edges,
    use_scale_free_distribution=True
)

In [11]:
negative_graph.get_number_of_directed_edges()

1557384

In [12]:
negative_graph.get_number_of_nodes()

1089125

In [13]:
negative_graph.dump_edges(
    DATA_PATH+'negative_edges.tsv',
    separator='\t',
    header = True,
    sources_column_number = 0,
    destinations_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

### Load embeddings

In [14]:
model_name = 'TransE_l1'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    transe_embedding_dict = pickle.load(filep)

In [15]:
model_name = 'ComplEx'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    complex_embedding_dict = pickle.load(filep)

In [16]:
model_name = 'DistMult'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    distmult_embedding_dict = pickle.load(filep)

### Load KG data as edges

In [20]:
filename = 'reference-data/reference_data_curie_edges_20240725.tsv'

In [21]:
refdf = pd.read_csv(DATA_PATH+filename, sep='\t')
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1364 entries, 0 to 1363
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   source            1364 non-null   object
 1   target            1364 non-null   object
 2   interaction_type  1364 non-null   object
dtypes: object(3)
memory usage: 32.1+ KB


In [22]:
negative_edges = pd.read_csv(DATA_PATH+'negative_edges.tsv', sep='\t')
negative_edges['relation'] = 'negative'
negative_edges = negative_edges[['subject', 'relation', 'object']]
negative_edges.head()

,subject,relation,object
0,chebi:172924,negative,uberon:0002046
1,go:0034378,negative,so:0000673
2,go:0034378,negative,hp:0000819
3,go:0034378,negative,pato:0000460
4,reactome:R-HSA-373707,negative,ncbigene:55755


In [23]:
kg_edges = pd.read_csv(KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v3.0.0.tsv', sep='\t')
kg_edges = kg_edges[['source', 'relation', 'target']]
kg_edges = kg_edges.rename(columns={'source': 'subject', 'target': 'object'})
kg_edges.head()

,subject,relation,object
0,mgi:1913508,rdfs:subClassOf,so:0001217
1,mgi:1913508,ro:0002160,ncbitaxon:10090
2,so:0001217,rdfs:subClassOf,so:0000704
3,so:0001217,ro:0000086,so:0000010
4,so:0001217,bfo:0000050,ncbitaxon:9606


In [24]:
kg_edges_combined = pd.concat([kg_edges, negative_edges], ignore_index=True)
kg_edges_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9344307 entries, 0 to 9344306
Data columns (total 3 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   subject   object
 1   relation  object
 2   object    object
dtypes: object(3)
memory usage: 213.9+ MB


In [25]:
kg_edges_combined.head()

,subject,relation,object
0,mgi:1913508,rdfs:subClassOf,so:0001217
1,mgi:1913508,ro:0002160,ncbitaxon:10090
2,so:0001217,rdfs:subClassOf,so:0000704
3,so:0001217,ro:0000086,so:0000010
4,so:0001217,bfo:0000050,ncbitaxon:9606


In [ ]:
##subset kg edges to only 'negative', 'inhibits' and 'positively regulates/stimulates' relations
kg_edges_combined = kg_edges_combined[kg_edges_combined['relation'].isin(['negative', 'ro:0002449', 'ro:0002213'])]

In [ ]:
##get counts
kg_edges_combined['relation'].value_counts()

In [26]:
le = LabelEncoder()
kg_edges_combined['relation_encoded'] = le.fit_transform(kg_edges_combined['relation'])

### Edge Type Prediction

In [27]:
def get_embeddings_df(embedding_dict):
    node_embeddings_df = embedding_dict['node_embeddings'][0]
    edge_type_embeddings_df = embedding_dict['edge_type_embeddings'][0]
    return node_embeddings_df, edge_type_embeddings_df

In [28]:
def extract_features(edges, node_embeddings_df, edge_type_embeddings_df, edge_feature_method='concat'):
    features = []
    labels = []
    for subj, obj, pred in edges:
        try:
            node_subj_embedding = node_embeddings_df.loc[subj].values
            node_obj_embedding = node_embeddings_df.loc[obj].values

            # Handle 'negative' predicate specially
            if pred == 'negative':
                edge_type_embedding = np.zeros(node_subj_embedding.shape)
            else:
                edge_type_embedding = edge_type_embeddings_df.loc[pred].values
            if edge_feature_method == 'concat':
                feature = np.concatenate([node_subj_embedding, edge_type_embedding, node_obj_embedding])
            elif edge_feature_method == 'hadamard':
                node_feature = np.multiply(node_subj_embedding,node_obj_embedding)  # Hadamard product
                feature = np.concatenate([node_feature, edge_type_embedding])
            features.append(feature)
            labels.append(pred)
        except KeyError:
            # Handle missing embeddings (optional: skip or assign default values)
            continue
    return np.array(features), np.array(labels)

In [29]:
model_name = 'TransE'
if model_name == 'TransE':
    embedding_dict = transe_embedding_dict
elif model_name == 'ComplEx':
    embedding_dict = complex_embedding_dict
elif model_name == 'DistMult':
    embedding_dict = distmult_embedding_dict

In [30]:
##try with gpu
node_embeddings_df, edge_type_embeddings_df = get_embeddings_df(embedding_dict)
edge_feature_method = 'concat'
# Prepare the data
kg_edges_array = kg_edges_combined[['subject', 'object', 'relation_encoded']].values
X_kg, y_kg = extract_features(kg_edges_array, node_embeddings_df, edge_type_embeddings_df, edge_feature_method)

print('TransE embeddings concatenated with edge type embeddings and edge type prediction with MLP classifier: ')
edge_type_classifier = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

edge_type_accuracies = []
edge_type_precisions = []
edge_type_recalls = []
edge_type_f1s = []
edge_type_aucs = []

for train_index, test_index in kf.split(X_kg):
    X_train, X_test = X_kg[train_index], X_kg[test_index]
    y_train, y_test = y_kg[train_index], y_kg[test_index]

    edge_type_classifier.fit(X_train, y_train)
    y_pred = edge_type_classifier.predict(X_test)
    y_proba = edge_type_classifier.predict_proba(X_test)
    edge_type_accuracies.append(accuracy_score(y_test, y_pred))
    edge_type_precisions.append(precision_score(y_test, y_pred, average='weighted'))
    edge_type_recalls.append(recall_score(y_test, y_pred, average='weighted'))
    edge_type_f1s.append(f1_score(y_test, y_pred, average='weighted'))
    edge_type_aucs.append(roc_auc_score(y_test, y_proba, multi_class='ovr'))

print("Edge-Type Prediction Accuracy: {:.2f} ± {:.2f}".format(np.mean(edge_type_accuracies), np.std(edge_type_accuracies)))
print("Edge-Type Prediction Precision: {:.2f} ± {:.2f}".format(np.mean(edge_type_precisions), np.std(edge_type_precisions)))
print("Edge-Type Prediction Recall: {:.2f} ± {:.2f}".format(np.mean(edge_type_recalls), np.std(edge_type_recalls)))
print("Edge-Type Prediction F1 Score: {:.2f} ± {:.2f}".format(np.mean(edge_type_f1s), np.std(edge_type_f1s)))
print("Edge-Type Prediction AUC: {:.2f} ± {:.2f}".format(np.mean(edge_type_aucs), np.std(edge_type_aucs)))

KeyboardInterrupt: 

### Predict edge types in reference data

In [ ]:
# Prepare the reference data
ref_edges = refdf[['source', 'target']].values
X_reference = extract_features_hadamard(ref_edges, node_embeddings_df, edge_type_embeddings_df)

# Predict edge types for the reference data
predicted_edge_types = edge_type_classifier.predict(X_reference)
refdf['predicted_edge_type'] = le.inverse_transform(predicted_edge_types)

# Save the predictions
refdf.to_csv('reference_edges_with_predictions.tsv', sep='\t', index=False)
